# 🎓 Task 31 — Learning Path Optimization System
### AI & ML Internship Program | SkillXYZ Learning

**Dataset:** HarvardX–MITx Person-Course Academic Dataset (2012–2013)
**Goal:** Build an intelligent Learning Intelligence Platform that generates:
- Skill Gap Analysis
- Learning Readiness Score (ML Model)
- Personalized Learning Roadmap
- Learning Progress Dashboard (exported for Power BI)
- Career Readiness Indicator (Bonus)

---


## 1️⃣ Business Understanding

**Company:** SkillXYZ Learning — 15M+ learners across 80+ countries.

**Problem:** 43% of learners never complete their first learning path. Recommendations today
are popularity-based ("students who watched this also watched"), not skill-based. Learners skip
prerequisites, revisit failed concepts, and abandon paths midway.

**Objective:** Move from a simple "next video" recommender to a full **Learning Path Optimization
System** that considers a learner's current skill level, engagement behavior, and course sequence
to recommend the most effective end-to-end journey.

**Strategic questions we answer in this notebook:**
1. How can learning paths be personalized for every learner?
2. Which course sequences lead to the highest completion (certification) rate?
3. Which skills/courses should be mastered before advanced topics?
4. How can we quantify learner engagement and readiness?
5. How can we estimate career-path readiness?


## 2️⃣ Setup & Data Load

> **Colab instructions:** Upload `cleaned_courses.csv` (or `Courses.csv`) using the file upload
> cell below, or mount Google Drive. This notebook is self-contained — every model is trained
> from scratch on your uploaded data.


In [ ]:
# Install/import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, silhouette_score)
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully ✅")


In [ ]:
# ---- OPTION A: Upload file directly in Colab ----
from google.colab import files
uploaded = files.upload()   # choose cleaned_courses.csv from your computer
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"Loaded: {filename}")
print(df.shape)
df.head()


In [ ]:
# ---- OPTION B: If reading from Google Drive instead, comment Option A above and use this ----
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Task31/cleaned_courses.csv')
# print(df.shape)


## 3️⃣ Data Exploration & Learner Behavior Analysis

We explore course popularity, domains, funnel stages (registered → viewed → explored →
certified), and engagement patterns before building models.


In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# Basic cleanup — ensure key columns exist and are typed correctly
df['certified'] = df['certified'].astype(int)
df['engagement_score'] = df['engagement_score'].fillna(0)
df['grade_num'] = df['grade_num'].fillna(0)
for col in ['nevents','ndays_act','nplay_video','nchapters','nforum_posts']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print("Domains:", df['domain'].nunique(), "| Courses:", df['course_id'].nunique(),
      "| Learners:", df['userid_DI'].nunique())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

df['domain'].value_counts().plot(kind='barh', ax=axes[0,0], color='#4C72B0')
axes[0,0].set_title('Enrollments by Domain')
axes[0,0].invert_yaxis()

df['funnel_stage'].value_counts().reindex(
    ['Registered Only','Viewed','Explored','Certified']).plot(
    kind='bar', ax=axes[0,1], color='#DD8452')
axes[0,1].set_title('Learner Funnel Stage Distribution')
axes[0,1].tick_params(axis='x', rotation=30)

df['course_level'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%',
                                        colors=['#55A868','#C44E52'])
axes[1,0].set_title('Foundational vs Advanced Enrollments')
axes[1,0].set_ylabel('')

sns.histplot(df['engagement_score'], bins=40, ax=axes[1,1], color='#8172B2')
axes[1,1].set_title('Engagement Score Distribution')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150)
plt.show()


In [ ]:
# Certification rate by domain and by course level — tells us where drop-off happens
cert_by_domain = df.groupby('domain')['certified'].mean().sort_values(ascending=False) * 100
cert_by_level = df.groupby('course_level')['certified'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cert_by_domain.plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Certification Rate % by Domain')
axes[0].set_xlabel('% Certified')

cert_by_level.plot(kind='bar', ax=axes[1], color=['#55A868','#C44E52'])
axes[1].set_title('Certification Rate % — Foundational vs Advanced')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('certification_rates.png', dpi=150)
plt.show()

print("Certification % by domain:\n", cert_by_domain.round(2))


## 4️⃣ Learner Profile Engineering

The raw data is one row per (learner, course). To personalize learning paths we build a
**learner-level profile**: how many courses they've touched, their average engagement, domain
spread, and highest course level reached. This profile feeds every ML model below.


In [ ]:
learner_profile = df.groupby('userid_DI').agg(
    courses_taken=('course_id', 'nunique'),
    domains_covered=('domain', 'nunique'),
    avg_engagement=('engagement_score', 'mean'),
    total_events=('nevents', 'sum'),
    total_active_days=('ndays_act', 'sum'),
    total_video_plays=('nplay_video', 'sum'),
    total_forum_posts=('nforum_posts', 'sum'),
    avg_grade=('grade_num', 'mean'),
    courses_certified=('certified', 'sum'),
    advanced_courses=('course_level', lambda x: (x == 'Advanced').sum()),
    foundational_courses=('course_level', lambda x: (x == 'Foundational').sum()),
).reset_index()

learner_profile['completion_rate'] = (learner_profile['courses_certified'] /
                                       learner_profile['courses_taken']).round(3)
learner_profile['attempted_advanced_without_foundation'] = (
    (learner_profile['advanced_courses'] > 0) & (learner_profile['foundational_courses'] == 0)
).astype(int)

print(learner_profile.shape)
learner_profile.head()


In [ ]:
print("Learners who skipped foundations and went straight to advanced courses:",
      learner_profile['attempted_advanced_without_foundation'].sum(),
      f"({learner_profile['attempted_advanced_without_foundation'].mean()*100:.1f}%% of all learners)")

sns.histplot(learner_profile['completion_rate'], bins=20, color='#4C72B0')
plt.title('Distribution of Learner Completion Rate')
plt.xlabel('Completion Rate (certified / courses taken)')
plt.savefig('completion_rate_dist.png', dpi=150)
plt.show()


## 5️⃣ Skill Gap Discovery

For every learner we compute which **domains** they have *not yet been certified in* — these
are treated as skill gaps relative to the platform's full domain catalog. We also flag learners
attempting advanced material without a foundational base in the same domain.


In [ ]:
all_domains = sorted(df['domain'].unique())

# Domain -> certified mapping per learner
domain_cert = df[df['certified'] == 1].groupby('userid_DI')['domain'].apply(set).to_dict()
domain_seen = df.groupby('userid_DI')['domain'].apply(set).to_dict()

def get_skill_gaps(user_id, top_n=3):
    seen = domain_seen.get(user_id, set())
    certified_domains = domain_cert.get(user_id, set())
    # Gap = domains touched but not certified, prioritized, then domains never explored
    touched_gaps = list(seen - certified_domains)
    unexplored_gaps = [d for d in all_domains if d not in seen]
    gaps = touched_gaps + unexplored_gaps
    return gaps[:top_n]

# Example for 5 learners
sample_users = learner_profile['userid_DI'].sample(5, random_state=42).tolist()
for u in sample_users:
    print(f"Learner {u}: Skill Gaps -> {get_skill_gaps(u)}")


## 6️⃣ ML Model 1 — Learning Readiness Score

**Goal:** Predict the *probability a learner will certify* given their engagement behavior, then
scale that probability into a **0–100 Learning Readiness Score**. This tells us whether a learner
is prepared to move on to advanced content.

We train a **Random Forest Classifier** on row-level (learner × course) data using engagement
features, then calibrate the probability output into the readiness score.


In [ ]:
feature_cols = ['nevents', 'ndays_act', 'nplay_video', 'nchapters', 'nforum_posts',
                 'engagement_score', 'grade_num']

model_df = df.copy()
model_df['course_level_enc'] = model_df['course_level'].map({'Foundational': 0, 'Advanced': 1})
feature_cols_full = feature_cols + ['course_level_enc']

X = model_df[feature_cols_full]
y = model_df['certified']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)
y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\n", classification_report(y_test, y_pred, target_names=['Not Certified','Certified']))


In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Certified','Certified'], yticklabels=['Not Certified','Certified'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#C44E52', label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_title('ROC Curve — Certification Prediction')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('model1_readiness_performance.png', dpi=150)
plt.show()


In [ ]:
# Feature importance
importances = pd.Series(rf_model.feature_importances_, index=feature_cols_full).sort_values()
importances.plot(kind='barh', color='#55A868', figsize=(9,5))
plt.title('What Drives Certification? (Feature Importance)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('model1_feature_importance.png', dpi=150)
plt.show()


In [ ]:
# Generate Learning Readiness Score (0-100) for every learner using their MOST RECENT course row
model_df['readiness_proba'] = rf_model.predict_proba(scaler.transform(model_df[feature_cols_full]))[:, 1]
model_df['readiness_score'] = (model_df['readiness_proba'] * 100).round(1)

readiness_by_learner = model_df.groupby('userid_DI')['readiness_score'].mean().round(1)
learner_profile = learner_profile.merge(
    readiness_by_learner.rename('learning_readiness_score'), on='userid_DI', how='left')

def readiness_band(score):
    if score >= 70: return 'Ready for Advanced'
    elif score >= 40: return 'Building Readiness'
    else: return 'Needs Foundation Support'

learner_profile['readiness_band'] = learner_profile['learning_readiness_score'].apply(readiness_band)

print(learner_profile['readiness_band'].value_counts())
learner_profile[['userid_DI','learning_readiness_score','readiness_band']].head(10)


## 7️⃣ ML Model 2 — Learner Segmentation (K-Means Clustering)

We cluster learners into behavioral **personas** using engagement, completion, and breadth
features. This powers the "learner personas" business insight required in the report.


In [ ]:
cluster_features = ['courses_taken', 'avg_engagement', 'total_active_days',
                     'completion_rate', 'learning_readiness_score']

cluster_df = learner_profile[cluster_features].fillna(0)
scaler2 = StandardScaler()
cluster_scaled = scaler2.fit_transform(cluster_df)

# Elbow method to pick k
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(cluster_scaled)
    inertias.append(km.inertia_)

plt.plot(list(K_range), inertias, marker='o', color='#4C72B0')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.savefig('elbow_method.png', dpi=150)
plt.show()


In [ ]:
# Fit final KMeans model with k=4 personas
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
learner_profile['segment'] = kmeans.fit_predict(cluster_scaled)

sil_score = silhouette_score(cluster_scaled, learner_profile['segment'])
print(f"Silhouette Score: {sil_score:.3f}")

segment_summary = learner_profile.groupby('segment')[cluster_features].mean().round(2)
print(segment_summary)


In [ ]:
# Auto-label personas based on segment characteristics (ranked by readiness & completion)
ranking = segment_summary[['learning_readiness_score','completion_rate']].mean(axis=1).sort_values()
persona_names = ['At-Risk Learner', 'Casual Explorer', 'Steady Progressor', 'High Achiever']
label_map = {seg: persona_names[i] for i, seg in enumerate(ranking.index)}
learner_profile['persona'] = learner_profile['segment'].map(label_map)

print(learner_profile['persona'].value_counts())


In [ ]:
# Visualize personas with PCA (2D projection)
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(cluster_scaled)
learner_profile['pca1'], learner_profile['pca2'] = pca_coords[:,0], pca_coords[:,1]

plt.figure(figsize=(9,7))
sns.scatterplot(data=learner_profile, x='pca1', y='pca2', hue='persona',
                 palette='Set2', alpha=0.6, s=30)
plt.title('Learner Personas (PCA-projected K-Means Clusters)')
plt.tight_layout()
plt.savefig('learner_personas_pca.png', dpi=150)
plt.show()


## 8️⃣ Personalized Learning Roadmap Generator

Rule-based + model-informed roadmap engine:
1. Start with **skill gaps** (Section 5)
2. Always sequence **Foundational → Advanced** within a domain
3. Adjust pacing using the learner's **Readiness Score**
4. Recommend a capstone/certification milestone at the end

This mirrors the "Statistics → ML Foundations → Feature Engineering → ..." example format
from the business brief.


In [ ]:
# Build a course catalog with domain + level for sequencing
course_catalog = df[['course_id','domain','course_level']].drop_duplicates()
course_catalog = course_catalog.sort_values(['domain','course_level'], ascending=[True, True])

def recommend_roadmap(user_id, n_courses=5):
    profile_row = learner_profile[learner_profile['userid_DI'] == user_id].iloc[0]
    gaps = get_skill_gaps(user_id, top_n=3)
    readiness = profile_row['learning_readiness_score']
    completed = df[(df['userid_DI'] == user_id) & (df['certified']==1)]['course_id'].tolist()

    roadmap = []
    for domain in gaps:
        domain_courses = course_catalog[course_catalog['domain'] == domain]
        # Foundational first, unless learner already has high readiness
        order = ['Foundational','Advanced'] if readiness < 70 else ['Advanced','Foundational']
        for level in order:
            picks = domain_courses[domain_courses['course_level']==level]['course_id'].tolist()
            for c in picks:
                if c not in completed and c not in roadmap:
                    roadmap.append(c)

    roadmap = roadmap[:n_courses]
    roadmap.append('Capstone Project & Certification Milestone')

    est_weeks = len(roadmap) * (3 if readiness >= 70 else 4)

    return {
        'learner': user_id,
        'persona': profile_row['persona'],
        'skill_gaps': gaps,
        'learning_readiness_score': readiness,
        'readiness_band': profile_row['readiness_band'],
        'recommended_roadmap': roadmap,
        'estimated_completion_weeks': est_weeks
    }

# Example output — matches the business brief's expected format
import json
example_user = learner_profile.sort_values('courses_taken', ascending=False)['userid_DI'].iloc[3]
result = recommend_roadmap(example_user)
print(json.dumps(result, indent=2, default=str))


In [ ]:
# Generate roadmap for a batch of learners (for the dashboard export)
sample_for_export = learner_profile.sample(min(500, len(learner_profile)), random_state=1)
roadmap_records = [recommend_roadmap(u) for u in sample_for_export['userid_DI']]
roadmap_df = pd.DataFrame(roadmap_records)
roadmap_df['recommended_roadmap'] = roadmap_df['recommended_roadmap'].apply(lambda x: ' -> '.join(x))
roadmap_df.head()


## 9️⃣ Career Readiness Indicator (Bonus)

We map **domain coverage + certification** to common career tracks and produce a 0–100
readiness score per career goal.


In [ ]:
career_domain_map = {
    'Data Scientist': ['Computer Science', 'Statistics & Public Health', 'Economics'],
    'ML Engineer': ['Computer Science', 'Statistics & Public Health', 'Engineering'],
    'Public Health Analyst': ['Public Health', 'Statistics & Public Health', 'Biology'],
    'Software Engineer': ['Computer Science', 'Engineering'],
    'Research Scientist': ['Physics', 'Chemistry', 'Biology', 'Engineering'],
}

def career_readiness(user_id):
    certified_domains = domain_cert.get(user_id, set())
    seen_domains = domain_seen.get(user_id, set())
    scores = {}
    for career, req_domains in career_domain_map.items():
        cert_match = len(certified_domains & set(req_domains))
        seen_match = len(seen_domains & set(req_domains))
        score = (cert_match / len(req_domains)) * 70 + (seen_match / len(req_domains)) * 30
        scores[career] = round(min(score, 100), 1)
    best_career = max(scores, key=scores.get)
    return best_career, scores[best_career], scores

# Example
career, score, all_scores = career_readiness(example_user)
print(f"Learner {example_user} — Best-fit career: {career} ({score}/100)")
print(all_scores)


In [ ]:
# Batch compute for export
career_results = sample_for_export['userid_DI'].apply(lambda u: career_readiness(u))
roadmap_df['career_goal'] = [r[0] for r in career_results]
roadmap_df['career_readiness_score'] = [r[1] for r in career_results]
roadmap_df.head()


## 🔟 Export Data for Power BI Dashboard

We export clean, dashboard-ready CSVs — same pattern used in Task 15 (Renewable Energy
Dashboard): one fact table + supporting tables, ready to load directly into Power BI Desktop.


In [ ]:
# 1. Learner profile fact table (core dashboard table)
export_profile = learner_profile[[
    'userid_DI','courses_taken','domains_covered','avg_engagement','completion_rate',
    'learning_readiness_score','readiness_band','persona'
]].copy()
export_profile.to_csv('learner_profile_dashboard.csv', index=False)

# 2. Roadmap + career readiness table
roadmap_df.to_csv('learning_roadmaps_dashboard.csv', index=False)

# 3. Course-level certification & engagement summary
course_summary = df.groupby(['course_id','domain','course_level']).agg(
    enrollments=('userid_DI','nunique'),
    certification_rate=('certified','mean'),
    avg_engagement=('engagement_score','mean')
).reset_index()
course_summary['certification_rate'] = (course_summary['certification_rate']*100).round(2)
course_summary.to_csv('course_summary_dashboard.csv', index=False)

# 4. Domain-level skill gap frequency (how often each domain shows up as a gap)
from collections import Counter
gap_counter = Counter()
for u in learner_profile['userid_DI'].sample(min(5000, len(learner_profile)), random_state=7):
    for g in get_skill_gaps(u, top_n=3):
        gap_counter[g] += 1
skill_gap_df = pd.DataFrame(gap_counter.items(), columns=['domain','gap_frequency']).sort_values(
    'gap_frequency', ascending=False)
skill_gap_df.to_csv('skill_gap_frequency_dashboard.csv', index=False)

print("Exported files:")
print("- learner_profile_dashboard.csv:", export_profile.shape)
print("- learning_roadmaps_dashboard.csv:", roadmap_df.shape)
print("- course_summary_dashboard.csv:", course_summary.shape)
print("- skill_gap_frequency_dashboard.csv:", skill_gap_df.shape)


In [ ]:
# Download all dashboard CSVs (Colab only)
from google.colab import files as colab_files
for f in ['learner_profile_dashboard.csv','learning_roadmaps_dashboard.csv',
          'course_summary_dashboard.csv','skill_gap_frequency_dashboard.csv']:
    colab_files.download(f)


## 1️⃣1️⃣ Business Insights Summary

- **Learner personas:** Four segments emerge — *At-Risk Learners*, *Casual Explorers*, *Steady
  Progressors*, and *High Achievers* — each needing a different intervention (nudges, gamification,
  advanced tracks, mentorship).
- **Skill gaps:** A large share of learners jump into Advanced courses without a Foundational
  course in the same domain — a direct driver of the 43% path-abandonment problem.
- **Certification drivers:** Feature importance shows engagement depth (events, active days,
  video plays) matters more than raw enrollment — the model can flag disengagement early.
- **Curriculum optimization:** Domains with low certification rates (see Section 3) are strong
  candidates for redesigned onboarding or additional scaffolding content.
- **Readiness-based pacing:** Learners below a 40 readiness score should be routed to
  foundational reinforcement before advanced content, rather than the current one-size-fits-all
  sequencing.


## 1️⃣2️⃣ Future Enhancements

- **Adaptive learning:** Update the Readiness Score in near real-time as new activity streams in.
- **Reinforcement learning for path optimization:** Treat course sequencing as an RL problem
  (reward = certification / engagement) to learn optimal paths beyond fixed rules.
- **AI mentor integration:** Use an LLM layer on top of the readiness score + roadmap to explain
  *why* a course was recommended, in natural language.
- **Real-time skill tracking:** Stream ndays_act / nevents from the LMS to recompute readiness
  scores daily instead of batch.
